# `c04_ef` — Fall Enrollment, Retention, and Student-to-Faculty Ratio

**Component curation notebook.** Fetches the raw IPEDS distribution files, verifies the
reference period against official documentation, locks the schema, reshapes to the
declared grain, validates, and writes one curated table with a metadata sidecar.

| Property | Value |
|---|---|
| Native tables | `EF2023A`, `EF2023B`, `EF2023C`, `EF2023D`, `DRVEF2023` |
| Reference period | Fall 2023 census date |
| Curated grain | `UNITID` x `EFALEVEL` |
| Output | `data/curated/c04_ef.parquet` |

Retention and student-to-faculty ratio live in EF2023D, at UNITID grain, and are the two most reused features in the modelling chapters. EF2023A is level by demographic detail; select the EFALEVEL rows you need rather than summing across them, because the file mixes totals with their own components.

> **Pitfall.** EF2023D reports RET_PCF for full-time and RET_PCP for part-time cohorts. They are different denominators and are not interchangeable; most published 'retention rate' figures mean RET_PCF, and silently substituting the part-time series produces a different and much noisier measure.

## 1. Environment

One import surface, so a parsing quirk is fixed once rather than twelve times.

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import ipeds_utils as iu

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore", category=FutureWarning)

SLUG = "c04_ef"
TABLES = ['EF2023A', 'EF2023B', 'EF2023C', 'EF2023D', 'DRVEF2023']
GRAIN = ['UNITID', 'EFALEVEL']
REFERENCE_PERIOD = 'Fall 2023 census date'

print("ipeds_utils", iu.__version__, "| pandas", pd.__version__)

ipeds_utils 1.1.0 | pandas 3.0.5


## 2. Retrieve

Downloads are cached, so re-running this notebook is offline and cheap. Every retrieval returns a provenance record carrying a SHA-256 digest, which is what makes a result reproducible rather than merely repeatable.

In [2]:
RAW_DIR = "../data/raw"   # relative to notebooks/, so all twelve share one cache

provenance = [iu.fetch(t, raw_dir=RAW_DIR) for t in TABLES]
pd.DataFrame(provenance)[["table", "data_bytes", "data_sha256", "retrieved_utc"]]

,table,data_bytes,data_sha256,retrieved_utc
0,EF2023A,2979525,d1fbd91cbda965e6bf75b9167825ec121d9ebf218ef7bc...,2026-09-24T17:19:16+00:00
1,EF2023B,1741540,1aa81bcff050d82920c2d4ab921b91c763e2f9ff77adc4...,2026-09-24T17:19:16+00:00
2,EF2023C,181241,add4ece19bb4f8a5b7a893f60c9f96411beed137ec62a4...,2026-09-24T17:19:16+00:00
3,EF2023D,94562,cfe4267b5b2e2a2ce6ea8c079c64707868c8c5b22372d5...,2026-09-24T17:19:16+00:00
4,DRVEF2023,415320,8d38adec5a7febeaa0060310534e1871d09621fe3620f0...,2026-09-24T17:19:16+00:00


## 3. Verify the reference period

**Do not skip this cell.** The filename year is not the reference period, and the offsets are not uniform across components. This assertion fails loudly rather than letting a misaligned period corrupt every downstream year comparison, where it would be invisible in the data itself.

In [3]:
intro = iu.assert_reference_period(
    provenance[0]["dict_path"],
    expect=r'(fall 2023|2023-24)',
    table=TABLES[0],
)
print(intro[:600])

File documentation for enrollment by race/ethnicity, gender, attendance status, and level of student: Fall 2023
(Provisional release)
Filename EF2023A
Overview This file contains the number of students enrolled in the fall, by race/ethnicity, gender, attendance (full- or part-time) status and level of student.  Institutions with traditional academic year calendar systems (semester, quarter, trimester or  4-1-4) report their enrollment as of October 15 or the official fall reporting date of the institution. Institutions with calendar systems that differ by program or allow continuous enrollment


## 4. Inspect the dictionary

Variable labels come from the published dictionary, never from memory. This is also where value sets are read, so categorical decoding is driven by the official codebook and a taxonomy revision surfaces as unmatched codes instead of a plausible-looking wrong label.

In [4]:
variables = iu.read_dict(provenance[0]["dict_path"])
valuesets = iu.read_valuesets(provenance[0]["dict_path"])

print(f"{len(variables)} variables documented, {len(valuesets)} value-set rows")
variables[["varname", "vartitle"]].head(20)

39 variables documented, 51 value-set rows


,varname,vartitle
0,UNITID,Unique identification number of the institution
1,EFALEVEL,Level of student
2,LINE,Level of student (original line number on surv...
3,SECTION,Attendance status of student
4,LSTUDY,Level of student
5,EFTOTLT,Grand total
6,EFTOTLM,Grand total men
7,EFTOTLW,Grand total women
8,EFAIANT,American Indian or Alaska Native total
9,EFAIANM,American Indian or Alaska Native men


## 5. Load and lock the schema

The first run records the column signature; later runs fail if it drifts.

In [5]:
KEEP = ['UNITID', 'EFALEVEL', 'EFTOTLT', 'EFTOTLM', 'EFTOTLW']

raw = iu.read_csv(provenance[0]["data_path"])
print("raw shape", raw.shape)

lock = iu.lock_schema(raw, TABLES[0], schema_dir="../schemas", strict=False)
print("schema:", lock["status"], "| added", lock["added"][:5], "| removed", lock["removed"][:5])

available = [c for c in KEEP if c in raw.columns]
missing = [c for c in KEEP if c not in raw.columns]
if missing:
    print("NOT PRESENT in this cycle (verify against the varlist above):", missing)

frame = raw[available].copy()
frame.head()

raw shape (115156, 73)
schema: unchanged | added [] | removed []


,UNITID,EFALEVEL,EFTOTLT,EFTOTLM,EFTOTLW
0,100654,1,6614,2671,3943
1,100654,2,5845,2417,3428
2,100654,3,5726,2389,3337
3,100654,4,1990,816,1174
4,100654,5,3736,1573,2163


## 6. Mask reserved missing codes

IPEDS encodes missingness as negative integers. A mean computed without masking them is badly wrong and looks entirely plausible, which is what makes this the most costly single omission in IPEDS analysis.

In [6]:
RESERVED = [-1, -2, -3, -9]

numeric_cols = [
    c for c in frame.columns
    if c not in ("UNITID", *GRAIN) and pd.api.types.is_numeric_dtype(frame[c])
]

before = frame[numeric_cols].isna().sum().sum()
for col in numeric_cols:
    frame.loc[frame[col].isin(RESERVED), col] = np.nan
after = frame[numeric_cols].isna().sum().sum()

# Masking turns an integer column into float (1 becomes 1.0). Measures can stay float,
# since NaN is what the models expect, but category codes go back to nullable integers
# so they print, join, and decode as codes rather than as 1.0.
for col in ['EFALEVEL']:
    if col in frame.columns and pd.api.types.is_float_dtype(frame[col]):
        if (frame[col].dropna() % 1 == 0).all():
            frame[col] = frame[col].astype("Int64")

print(f"masked {after - before:,} reserved-code cells across {len(numeric_cols)} numeric columns")

masked 0 reserved-code cells across 3 numeric columns


## 7. Carry the imputation flags

An imputed value and a reported value are not the same evidence. A column where most institutions carry a generated flag should not be modelled as though it were observed, and this is where that judgement becomes possible.

In [7]:
values, flags = iu.split_imputation_flags(raw, numeric_cols)

if flags.shape[1] > 1:
    summary = iu.imputation_summary(flags)
    display(summary.head(15))
    reported = summary[summary.flag == "R"].set_index("column")["share"]
    weak = reported[reported < 0.90]
    if len(weak):
        print("Columns under 90% reported — interpret with care:")
        display(weak)
else:
    print("No X-prefixed imputation flags accompany this file.")

,column,flag,n,share
3,XEFTOTLM,R,112432,0.9763
4,XEFTOTLM,Z,2605,0.0226
5,XEFTOTLM,P,101,0.0009
6,XEFTOTLM,N,18,0.0002
0,XEFTOTLT,R,115037,0.9990
1,XEFTOTLT,P,101,0.0009
2,XEFTOTLT,N,18,0.0002
7,XEFTOTLW,R,113156,0.9826
8,XEFTOTLW,Z,1881,0.0163
9,XEFTOTLW,P,101,0.0009


## 8. Decode categoricals

Labels from the published value sets, not hand-typed mappings.

In [8]:
CATEGORICALS = ['EFALEVEL']

unresolved = {}
for col in CATEGORICALS:
    if col in frame.columns:
        frame = iu.decode(frame, valuesets, col)
        unmatched = frame.loc[frame[col].notna() & frame[f"{col}_LABEL"].isna(), col].unique()
        if len(unmatched):
            unresolved[col] = sorted(unmatched.tolist())[:10]

# An unmatched code means a taxonomy change or a parsing fault. Either way the
# labels are wrong, so this stops the notebook rather than printing a warning.
assert not unresolved, f"codes absent from the published value set: {unresolved}"

label_cols = [c for c in frame.columns if c.endswith("_LABEL")]
frame[CATEGORICALS + label_cols].drop_duplicates().head(20) if label_cols else frame.head()

,EFALEVEL,EFALEVEL_LABEL
0,1,All students total
1,2,"All students, Undergraduate total"
2,3,"All students, Undergraduate, Degree/certificat..."
3,4,"All students, Undergraduate, Degree/certificat..."
4,5,"All students, Undergraduate, Other degree/cert..."
5,11,"All students, Undergraduate, Non-degree/certif..."
6,12,"All students, Graduate"
7,19,"All students, Undergraduate, Other degree/cert..."
8,20,"All students, Undergraduate, Other degree/cert..."
9,21,Full-time students total


## 9. Reshape to the declared grain

Target grain: `UNITID` x `EFALEVEL`. The grain is asserted, not assumed, because a duplicated key silently inflates every aggregate computed downstream.

In [9]:
curated = frame.copy()

# This component already arrives at its declared grain, so curation is a
# pass-through. Components with a long layout (GRTYPE, EFFYALEV, STAFFCAT,
# OMCHRT) filter or pivot here instead; see c10_f for a full worked reshape.

present_grain = [g for g in GRAIN if g in curated.columns]
duplicated = curated.duplicated(subset=present_grain, keep=False).sum()
print(f"grain {present_grain} -> {len(curated):,} rows, {duplicated} duplicated")
assert duplicated == 0, "Declared grain is not unique; resolve before continuing."

curated.head()

grain ['UNITID', 'EFALEVEL'] -> 115,156 rows, 0 duplicated


,UNITID,EFALEVEL,EFTOTLT,EFTOTLM,EFTOTLW,EFALEVEL_LABEL
0,100654,1,6614.0,2671.0,3943.0,All students total
1,100654,2,5845.0,2417.0,3428.0,"All students, Undergraduate total"
2,100654,3,5726.0,2389.0,3337.0,"All students, Undergraduate, Degree/certificat..."
3,100654,4,1990.0,816.0,1174.0,"All students, Undergraduate, Degree/certificat..."
4,100654,5,3736.0,1573.0,2163.0,"All students, Undergraduate, Other degree/cert..."


## 10. Validate

Rules are declarative so the output is a persistable report: which checks ran, which failed, on how many rows, and which institutions were implicated. That report is the artefact you cite when claiming this table is fit for analysis.

In [10]:
RULES = [
    iu.unique_key('UNITID', 'EFALEVEL'),
    iu.in_range('EFTOTLT', 0, None),
]

report = iu.validate(curated, RULES, SLUG)
report.save(f"../reports/validation/{SLUG}.json")
display(report.to_frame()[["name", "status", "n_offending", "share", "note"]])

print("PASSED" if report.ok else "FAILED")
report.raise_if_failed()

,name,status,n_offending,share,note
0,"unique_key(UNITID,EFALEVEL)",pass,0,0.0,Declared grain must be unique
1,"in_range(EFTOTLT,0,None)",pass,0,0.0,Value plausibility bound


PASSED


Report(table='c04_ef', rows=115156, results=[{'name': 'unique_key(UNITID,EFALEVEL)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Declared grain must be unique', 'status': 'pass'}, {'name': 'in_range(EFTOTLT,0,None)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Value plausibility bound', 'status': 'pass'}], generated_utc='2026-09-24T17:19:17+00:00')

## 11. Write the curated table

The sidecar carries the reference period and grain with the data. This is the defence against assembling a panel by filename year when the underlying periods are offset differently per component.

In [11]:
path = iu.write_curated(
    curated,
    SLUG,
    root="../data/curated",
    reference_period=REFERENCE_PERIOD,
    grain=GRAIN,
    provenance=provenance,
    notes="EF2023D reports RET_PCF for full-time and RET_PCP for part-time cohorts. They are different denominators and are not interchangeable; most published 'retention rate' figures mean RET_PCF, and silently substituting the part-time series produces a different and much noisier measure.",
)

iu.write_provenance(provenance, f"../docs/provenance/{SLUG}.json")
print("wrote", path, f"({len(curated):,} rows x {curated.shape[1]} columns)")

wrote ../data/curated/c04_ef.parquet (115,156 rows x 6 columns)


## 12. Exercises

1. Re-run this notebook against the prior collection cycle by changing `TABLES`. The schema lock and the period assertion will both object; resolve each objection and record what changed between cycles.
2. Identify the three columns with the lowest reported-flag share, and argue whether each belongs in a predictive model at all.
3. Construct one derived cross-tabulation from this table, then apply `iu.suppress` and `iu.k_anonymity` to it. Report the smallest equivalence class before and after coarsening, and state the k you would require before publishing.
4. EF2023D reports RET_PCF for full-time and RET_PCP for part-time cohorts. Write a validation rule that would catch this error if a colleague made it, and add it to `RULES` above.